# 16. Stacking Feature Ablation Study

**Tujuan:** Evaluasi Stacking Ensemble pada berbagai jumlah fitur (Full/Top-20/Top-15/Top-10/Top-5).
Apakah ensemble lebih toleran terhadap reduksi fitur dibanding single model?

**Input:** `cleaned_100.pkl`, `stacking_baseline_15.pkl`, `ablation_results_04.pkl`

**Output:** `stacking_ablation_16.pkl`, PNG visualisasi comparison

In [ ]:
import numpy as np
import pandas as pd
import pickle
import os
import time
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import f1_score, matthews_corrcoef, accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import warnings
warnings.filterwarnings('ignore')

RANDOM_SEED = 42
TEST_SIZE = 0.20
DATA_DIR = '../data/'

print('Libraries loaded.')

## 1. Load Data & Feature Ranking

In [ ]:
# Load cleaned dataset
with open(os.path.join(DATA_DIR, 'cleaned_100.pkl'), 'rb') as f:
    data = pickle.load(f)

X = data['X']
y = data['y']
feature_names = data.get('feature_names', [f'f{i}' for i in range(X.shape[1])])

# Load stacking baseline (for feature ranking from XGBoost)
with open(os.path.join(DATA_DIR, 'stacking_baseline_15.pkl'), 'rb') as f:
    stack_data = pickle.load(f)

# Load previous single-model ablation for comparison
with open(os.path.join(DATA_DIR, 'ablation_results_04.pkl'), 'rb') as f:
    single_ablation = pickle.load(f)

# Get feature ranking from XGBoost base model
xgb_model = stack_data['base_models']['XGBoost']
importances = xgb_model.feature_importances_
ranked_indices = np.argsort(importances)[::-1]
ranked_features = [feature_names[i] for i in ranked_indices]

print(f'Dataset: {X.shape[0]} samples, {X.shape[1]} features')
print(f'Top-5 features: {ranked_features[:5]}')

# Train/Test split (same seed as notebook 15)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y
)

## 2. Define Stacking Function

In [ ]:
def train_stacking(X_tr, X_te, y_tr, y_te, n_classes, config_name):
    """
    Train full stacking pipeline and return metrics.
    """
    base_learners = {
        'XGBoost': XGBClassifier(
            max_depth=6, n_estimators=100, learning_rate=0.1,
            use_label_encoder=False, eval_metric='mlogloss',
            random_state=RANDOM_SEED, verbosity=0
        ),
        'LightGBM': LGBMClassifier(
            max_depth=6, n_estimators=100, learning_rate=0.1,
            random_state=RANDOM_SEED, verbose=-1
        ),
        'CatBoost': CatBoostClassifier(
            depth=6, iterations=100, learning_rate=0.1,
            random_seed=RANDOM_SEED, verbose=0
        )
    }
    
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
    meta_train = np.zeros((X_tr.shape[0], n_classes * 3))
    meta_test = np.zeros((X_te.shape[0], n_classes * 3))
    
    start_time = time.time()
    
    for idx, (name, model) in enumerate(base_learners.items()):
        oof_preds = np.zeros((X_tr.shape[0], n_classes))
        test_preds = np.zeros((X_te.shape[0], n_classes))
        
        for train_idx, val_idx in cv.split(X_tr, y_tr):
            Xt, Xv = X_tr[train_idx], X_tr[val_idx]
            yt, yv = y_tr[train_idx], y_tr[val_idx]
            
            m = model.__class__(**model.get_params())
            m.fit(Xt, yt)
            oof_preds[val_idx] = m.predict_proba(Xv)
            test_preds += m.predict_proba(X_te) / cv.n_splits
        
        col_s = idx * n_classes
        col_e = (idx + 1) * n_classes
        meta_train[:, col_s:col_e] = oof_preds
        meta_test[:, col_s:col_e] = test_preds
    
    # Meta-learner
    scaler = StandardScaler()
    meta_tr_s = scaler.fit_transform(meta_train)
    meta_te_s = scaler.transform(meta_test)
    
    meta_lr = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED, multi_class='multinomial')
    meta_lr.fit(meta_tr_s, y_tr)
    y_pred = meta_lr.predict(meta_te_s)
    
    total_time = time.time() - start_time
    
    # Inference time (predict only)
    inf_times = []
    for _ in range(10):
        t0 = time.time()
        # Simulate full stacking inference
        test_meta = np.zeros((X_te.shape[0], n_classes * 3))
        for idx2, (nm, mdl) in enumerate(base_learners.items()):
            mdl_final = mdl.__class__(**mdl.get_params())
            mdl_final.fit(X_tr, y_tr)  # simplified
            test_meta[:, idx2*n_classes:(idx2+1)*n_classes] = mdl_final.predict_proba(X_te)
        meta_lr.predict(scaler.transform(test_meta))
        inf_times.append(time.time() - t0)
    avg_inf = np.mean(inf_times)
    
    return {
        'config': config_name,
        'n_features': X_tr.shape[1],
        'accuracy': accuracy_score(y_te, y_pred),
        'f1_score': f1_score(y_te, y_pred, average='weighted'),
        'mcc': matthews_corrcoef(y_te, y_pred),
        'train_time': total_time,
        'inference_time': avg_inf
    }

print('Stacking function defined.')

## 3. Run Ablation Study

In [ ]:
n_classes = len(np.unique(y_train))

feature_configs = {
    'Full': ranked_indices.tolist(),
    'Top-20': ranked_indices[:20].tolist(),
    'Top-15': ranked_indices[:15].tolist(),
    'Top-10': ranked_indices[:10].tolist(),
    'Top-5': ranked_indices[:5].tolist(),
}

ablation_results = []

print('='*70)
print('  STACKING FEATURE ABLATION STUDY')
print('='*70)

for config_name, feat_idx in feature_configs.items():
    print(f'\n  Config: {config_name} ({len(feat_idx)} features)')
    X_tr_sub = X_train[:, feat_idx]
    X_te_sub = X_test[:, feat_idx]
    
    result = train_stacking(X_tr_sub, X_te_sub, y_train, y_test, n_classes, config_name)
    ablation_results.append(result)
    
    print(f'    F1={result["f1_score"]*100:.2f}% | MCC={result["mcc"]:.4f} | Time={result["train_time"]:.1f}s')

ablation_df = pd.DataFrame(ablation_results)
print('\n' + '='*70)
print(ablation_df[['config', 'n_features', 'f1_score', 'mcc', 'train_time']].to_string(index=False))
print('='*70)

## 4. Comparison: Stacking vs Single Model Ablation

In [ ]:
print('\n' + '='*70)
print('  COMPARISON: Single XGBoost vs Stacking Ensemble (per config)')
print('='*70)
print(f'{"Config":<10} | {"Single F1%":<12} | {"Stacking F1%":<14} | {"Gain":<8}')
print('-'*50)

# Ambil F1 dari single model ablation (notebook 04)
# Format tergantung struktur pkl — adjust jika perlu
for res in ablation_results:
    cfg = res['config']
    stack_f1 = res['f1_score'] * 100
    # Placeholder untuk single model F1 (dari notebook 04)
    single_f1 = 98.0  # akan di-replace dengan data aktual
    gain = stack_f1 - single_f1
    print(f'{cfg:<10} | {single_f1:<12.2f} | {stack_f1:<14.2f} | {gain:+.2f}%')

## 5. Visualisasi

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

configs = ablation_df['config'].tolist()
x = np.arange(len(configs))

# F1-Score
axes[0].bar(x, ablation_df['f1_score']*100, color='steelblue', alpha=0.8)
axes[0].set_ylabel('F1-Score (%)')
axes[0].set_title('Stacking Ensemble F1 vs Feature Count')
axes[0].set_xticks(x)
axes[0].set_xticklabels(configs)
axes[0].set_ylim(90, 101)
axes[0].grid(axis='y', alpha=0.3)

# MCC
axes[1].bar(x, ablation_df['mcc'], color='orange', alpha=0.8)
axes[1].set_ylabel('MCC')
axes[1].set_title('Stacking Ensemble MCC vs Feature Count')
axes[1].set_xticks(x)
axes[1].set_xticklabels(configs)
axes[1].set_ylim(0.85, 1.0)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'stacking_ablation_f1_mcc.png'), bbox_inches='tight')
plt.show()
print('Saved: stacking_ablation_f1_mcc.png')

## 6. Save Results

In [ ]:
output = {
    'ablation_results': ablation_results,
    'ablation_df': ablation_df,
    'feature_configs': feature_configs,
    'ranked_indices': ranked_indices,
    'ranked_features': ranked_features
}

with open(os.path.join(DATA_DIR, 'stacking_ablation_16.pkl'), 'wb') as f:
    pickle.dump(output, f)

print('Saved: stacking_ablation_16.pkl')
print('\nNotebook 16 selesai. Lanjut ke 17 (Adversarial Attack pada Stacking).')